In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


E:\anaconda\Lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


True
NVIDIA GeForce GTX 1050 Ti


# after modifying overfitted model

In [2]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
from PIL import Image
import timm  # Library for Vision Transformers
from torch.cuda.amp import GradScaler, autocast  # Mixed precision training

# Check for CUDA
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Custom model with Swin Transformer
class CUBModel(nn.Module):
    def __init__(self, num_classes=200, pretrained=True):
        super(CUBModel, self).__init__()
        # Use Swin Transformer instead of ViT
        self.model = timm.create_model("swin_tiny_patch4_window7_224", pretrained=pretrained, num_classes=num_classes)
        self.dropout = nn.Dropout(p=0.3)  # Dropout for better generalization

    def forward(self, x):
        x = self.dropout(self.model(x))
        return x

# Training and Validation function
def train_and_validate(model, train_loader, val_loader, num_epochs, model_save_path):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  # Label smoothing to reduce overconfidence
    optimizer = AdamW(model.parameters(), lr=0.00003, weight_decay=0.01)  # Lower LR for stability
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    scaler = GradScaler()  # Mixed precision training

    best_val_acc = 0.0
    early_stopping_patience = 5  # Stop if no improvement for 5 epochs
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        correct_train = 0
        total_train = 0

        print(f"\nEpoch {epoch+1}/{num_epochs}")
        for images, labels in tqdm(train_loader, desc="Training", leave=False):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            # Mixed precision training
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total_train += labels.size(0)
            correct_train += predicted.eq(labels).sum().item()

        scheduler.step()

        train_acc = 100 * correct_train / total_train
        train_loss /= total_train

        # Validation
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validating", leave=False):
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                total_val += labels.size(0)
                correct_val += predicted.eq(labels).sum().item()

        val_acc = 100 * correct_val / total_val
        val_loss /= total_val

        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

        # Save model checkpoint if it's the best so far
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_no_improve = 0
            model_save_name = f"{model_save_path}_best.pth"
            torch.save(model.state_dict(), model_save_name)
            print(f"Best model saved to {model_save_name}")
        else:
            epochs_no_improve += 1

        # Early stopping
        if epochs_no_improve >= early_stopping_patience:
            print("Early stopping triggered")
            break

# Prediction function
def predict_image(model, image_path, class_names):
    transform = transforms.Compose([
        transforms.Resize((224, 224)), 
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        output = model(image_tensor)
        probabilities = torch.nn.functional.softmax(output[0], dim=0)
        top_prob, top_class = torch.topk(probabilities, 1)

    predicted_class = class_names[top_class.item()]
    predicted_probability = top_prob.item()

    print(f"Predicted Class: {predicted_class} | Probability: {predicted_probability:.4f}")
    return predicted_class, predicted_probability

# Data loading
data_dir = "E:\\kt project 2 Bird classification\\New version birds\\CUB_200_2011\\images"

# Data Augmentation
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomErasing(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_dataset = ImageFolder(root=data_dir, transform=transform_train)

# Train-Validation Split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Use validation transforms for validation dataset
val_dataset.dataset.transform = transform_val

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# Initialize the model
model = CUBModel(num_classes=len(full_dataset.classes)).to(device)

# Start training and save the model
train_and_validate(model, train_loader, val_loader, num_epochs=50, model_save_path="swin_model")

# Predict with a single image
model_path = "swin_model_best.pth"
image_path = "E:\\owl.webp"
class_names = full_dataset.classes

loaded_model = CUBModel(num_classes=len(class_names)).to(device)
loaded_model.load_state_dict(torch.load(model_path))
predict_image(loaded_model, image_path, class_names)


E:\anaconda\Lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Using device: cuda


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

E:\anaconda\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\harsh\.cache\huggingface\hub\models--timm--swin_tiny_patch4_window7_224.ms_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\harsh\AppData\Local\Temp\ipykernel_12540\1615316795.py:34: FutureWarning: `torch.cuda.amp.GradScaler(args.


Epoch 1/50


Training:   0%|                                                                                | 0/590 [00:00<?, ?it/s]C:\Users\harsh\AppData\Local\Temp\ipykernel_12540\1615316795.py:53: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


... (25 outputs left out)

Epoch 26/50 - Train Loss: 2.0617, Train Acc: 69.51% | Val Loss: 1.8076, Val Acc: 83.29%

Epoch 27/50


Epoch 27/50 - Train Loss: 2.0206, Train Acc: 70.56% | Val Loss: 1.7682, Val Acc: 83.21%

Epoch 28/50


Epoch 28/50 - Train Loss: 2.0047, Train Acc: 70.91% | Val Loss: 1.7872, Val Acc: 83.25%
Early stopping triggered


FileNotFoundError: [Errno 2] No such file or directory: 'E:\\owl.webp'

In [4]:
def user_predict(image_path):
    # Load the model
    model_path = "swin_model_best.pth"  # Change to your model path
    class_names = full_dataset.classes

    loaded_model = CUBModel(num_classes=len(class_names)).to(device)
    loaded_model.load_state_dict(torch.load(model_path))
    loaded_model.eval()

    # Transform the input image
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    # Load and preprocess the image
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)

    # Predict the class
    with torch.no_grad():
        output = loaded_model(image_tensor)
        probabilities = torch.nn.functional.softmax(output[0], dim=0)
        top_prob, top_class = torch.topk(probabilities, 1)

    predicted_class = class_names[top_class.item()]
    predicted_probability = top_prob.item()

    print(f"Predicted Class: {predicted_class} | Probability: {predicted_probability:.4f}")
    return predicted_class, predicted_probability


In [6]:
image_path = "E:\\kt project 2 Bird classification\\New version birds\\CUB_200_2011\\images\\200.Common_Yellowthroat\\Common_Yellowthroat_0018_190904.jpg"
user_predict(image_path)


Predicted Class: 200.Common_Yellowthroat | Probability: 0.7289


('200.Common_Yellowthroat', 0.7289261221885681)

# Views code

In [ ]:
from django.shortcuts import render
from django.http import JsonResponse
from PIL import Image
import torch
import torchvision.transforms as transforms
import timm

from django.shortcuts import render
from django.http import JsonResponse
from PIL import Image
import torch
import torchvision.transforms as transforms
import timm
import json
import torch.nn as nn

# Check for CUDA
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load cleaned class names from the JSON file
with open("D:\\Bird_class\\myapp\\cleaned_classes.json", "r") as file:
    class_names = json.load(file)

# Custom model with Swin Transformer
class CUBModel(nn.Module):
    def __init__(self, num_classes=200, pretrained=True):
        super(CUBModel, self).__init__()
        self.model = timm.create_model("swin_tiny_patch4_window7_224", pretrained=pretrained, num_classes=num_classes)
        self.dropout = nn.Dropout(p=0.3)

    def forward(self, x):
        x = self.dropout(self.model(x))
        return x

# Load the trained model
model_path = "D:\\Bird_class\\myapp\\swin_model_best.pth"
model = CUBModel(num_classes=len(class_names)).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

# Image transformation
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def predict_view(request):
    if request.method == "POST" and request.FILES.get("image"):
        # Load and preprocess the image
        image = Image.open(request.FILES["image"]).convert("RGB")
        image_tensor = transform(image).unsqueeze(0).to(device)

        # Perform prediction
        with torch.no_grad():
            output = model(image_tensor)
            probabilities = torch.nn.functional.softmax(output[0], dim=0)
            top_prob, top_class = torch.topk(probabilities, 1)

        predicted_class = class_names[top_class.item()]
        predicted_probability = top_prob.item()

        return JsonResponse({"class": predicted_class, "probability": f"{predicted_probability:.4f}"})

    return render(request, "D:\\Bird_class\\myapp\\templates\\myapp\\predict.html")



In [9]:
import json

def clean_class_names(file_path):
    class_names = []
    with open(file_path, "r") as file:
        for line in file:
            # Split each line by space and take the second part (bird name)
            parts = line.strip().split(" ", 1)
            if len(parts) == 2:
                class_name = parts[1]  # Get the bird name part
                class_names.append(class_name)
    return class_names

# Save the cleaned class names to a JSON file for quick access
def save_class_names(cleaned_class_names, save_path="cleaned_classes.json"):
    with open(save_path, "w") as file:
        json.dump(cleaned_class_names, file)

# Example usage
file_path = "classes.txt"  # Change to your file path
cleaned_class_names = clean_class_names(file_path)
save_class_names(cleaned_class_names)
print(f"Cleaned class names saved to cleaned_classes.json")


Cleaned class names saved to cleaned_classes.json
